# 06 · Explainability audit — what the pipeline already tells us (and we never read)

*Edge-features arc · 06 · machinery: `ridge_pipeline_throughline.ipynb`*

**The seam.** Our pipeline is a walk-forward that **refits ~407 times**, so every explainable artifact —
the legible base coefficients `coefs[i]`, the L1 survivor mask `masks[i]`, the regime-EBM shapes, the d8
importances — is actually a **time series of length 407** that we have always collapsed to one aggregate
number. We have only ever read the *cross-section* of our explanations, never the *time axis*. This chapter
mines two of those for the first time (coefficient trajectory, learned gate) and maps the rest.

*Well-timed:* the regime-MoE (ch. 04) and the kNN (ch. 05) both lose to the EBM → the lever is not another
model, it is **reading the explanations the pipeline already emits**.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))
LADDER = REPO / "results" / "moe_ladder"
print("setup ok")

---
## 1 · The close-damping coefficient *over time* — the 0DTE/OPEX dilution test

The legible `har_ma_5×close` coefficient (the −0.05 sign-flip, ch. 01) exists **per cadence block**. The
notes *hypothesize* (intraday_regime_findings §12) that daily 0DTE expiries diluted the monolithic close
effect. We can test it **directly**: does the close-damping coefficient trend toward zero across the sample?
`coef_trajectory.py` dumps the trajectory (writes `results/moe_ladder/coef_trajectory.csv`).

*Caveat:* row→date alignment is unresolved (§10), so we index by block (monotone in time); the trajectory
**shape** is the signal.

In [ ]:
p = LADDER / "coef_trajectory.csv"
if p.exists() and p.stat().st_size > 0:
    ct = pd.read_csv(p)
    cols = [c for c in ct.columns if c not in ("block", "row")]
    close_cols = [c for c in cols if "close" in c.lower() and "har" in c.lower()]
    print("tracked coef columns:", cols)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    for c in (close_cols or cols[:3]):
        y = ct[c].rolling(20, min_periods=1).mean()  # 20-block smooth to see the trend
        ax.plot(ct.block, y, label=c, lw=1.6)
    ax.axhline(0, c="0.4", lw=1, ls="--")
    ax.set_xlabel("cadence block (→ time)"); ax.set_ylabel("base coefficient (20-block MA)")
    ax.set_title("Does the close-damping coefficient weaken over the sample? (0DTE dilution test)")
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()
    for c in (close_cols or cols):
        d = ct[c].tail(10).mean() - ct[c].head(10).mean()
        print(f"  {c:30s} first10={ct[c].head(10).mean():+.4f}  last10={ct[c].tail(10).mean():+.4f}  Δ={d:+.4f}")
else:
    print("PENDING - run explain.sbatch (coef_trajectory.py) and sync results/moe_ladder/coef_trajectory.csv")

**Read:** a trajectory trending **toward zero** (late-sample |coef| < early) is direct evidence of the
0DTE/OPEX dilution — the close regime weakening as daily expiries spread the gamma unwind. A flat trajectory
says the regime is stationary and the floor is not an era artifact. *(interpretation folds in once the
trajectory renders.)*

---
## 2 · The learned MoE gate hyperplane — does it recover `hour`?

The gate is the **automated** version of ch. 01's hand-discovery (gradient descent over the partition,
across all features, vs the analyst naming `hour`). `gate_readout.py` re-fits one block's bagged gate and
dumps the decision-node weights vs feature names; `explain.sbatch` runs depth 1 and 2.

In [ ]:
from src.models.regime_moe import SoftTreeGate
import inspect, textwrap
from IPython.display import Markdown
print(textwrap.dedent(inspect.getsource(SoftTreeGate.forward)))

g = LADDER / "gate_readout.txt"   # paste/parse of logs/explain_*.out (top |w| features per gate node)
if g.exists():
    print(g.read_text())
else:
    print("\nPENDING - gate readout in logs/explain_*.out; sync as results/moe_ladder/gate_readout.txt")

**Read:** if the top-|w| splitter is **`hour`** (or the close indicator), the learned gate *recovers
the clock mechanism* — independent validation of the hand-discovery. If it splits on a **vol-level** or a
**HAR-state** feature instead, it found a partition the clock smeared (the one outcome the single-axis hand
loop cannot reach). Either way it is interpretable signal **even though** the MoE QLIKE (ch. 04) does not
beat the EBM — the payoff the architecture was for.

---
## 3 · The remaining underexploited probes (the map)

Mined above: #1 coefficient trajectory, #2 gate. Still on the table — all **pipeline-native**, needing no
new model:

| # | what the pipeline emits | the untested question | cost |
|---|---|---|---|
| 3 | `masks[i]` — the L1 survivor set per block | *which* ~120 features matter *when*? does reliance shift HAR→cumrv→exog around COVID? a free **regime-change detector** | cache read |
| 4 | the d8 global tree per block | **which pairs/triples** carry the "~90% order-≥3 interaction"? (Friedman H-stat / SHAP-interaction) — concentrated (→ distillable feature) or diffuse (→ needs a learner)? | refit + H-stat |
| 5 | the final cascade preds (saved) | what is **left** after base+global+regime? is the final residual white by hour/era/autocorr (floor real) or is there a missed axis? | preds read |
| 6 | the regime EBM's 4 bags | per-row **bag disagreement** = a *where-is-the-model-guessing* confidence map (reported only as aggregate SNR) | EBM refit |

The throughline: #3 and #5 need only *reading artifacts the walk-forward already produces*. #4 is the most
actionable for modelling — it tells us whether the high-order structure is concentrated enough that an
explicit feature (not a black-box learner) could distill more of the tree, closing the loop back to ch. 01's
distillation method. **The lever is mining what we have, not fitting something new.**

---
## Provenance
- `coef_trajectory.py` (per-block base coefs → `results/moe_ladder/coef_trajectory.csv`),
  `gate_readout.py` (one block's bagged gate → `logs/explain_*.out`), both run by `explain.sbatch` on
  cell `xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim`.
- Probes #3–#6 are specified, not yet built — each is a thin reader over the cache / saved preds.